In [1]:
import pandas as pd
import networkx as nx
import numpy as np
import pickle

import sys
sys.path.append('..')

# Loading Data

In [2]:
df_pre_graphed = pd.read_csv(f"../data_in_processing/E1S1_scaled.csv", index_col=[0, 1])

In [3]:
df_pre_graphed

Nuclear_size  ERKKTR_ratio  FoxO3A_ratio  \
Image_Metadata_T track_id                                             
0                1              303.000      0.704407       1.33383   
                 2              268.001      1.227760       1.20592   
                 3              370.000      0.779226       1.27561   
                 4              341.000      0.870780       1.34460   
                 5              308.000      0.807262       1.47321   
...                                 ...           ...           ...   
257              2849           165.000      0.607956       1.31242   
                 2850           138.000      0.775133       1.15722   
                 2855           148.000      1.114600       1.25315   
                 2856           148.000      0.861735       1.16059   
                 2857           175.000      0.589269       1.31594   

                           objNuclei_Location_Center_X  \
Image_Metadata_T track_id                                
0                1                             932.211   
                 2                             162.328   
                 3                             647.816   
                 4                             642.988   
                 5                             990.718   
...                                                ...   
257              2849                          839.988   
                 2850                         1019.510   
                 2855                          411.797   
                 2856                         1018.580   
                 2857                         1018.850   

                           objNuclei_Location_Center_Y  ERKKTR_ratio_scaled  \
Image_Metadata_T track_id                                                     
0                1                            875.2480             0.239322   
                 2                            365.4140             0.540105   
                 3                            846.5760             0.156325   
                 4                            827.8650             0.364739   
                 5                             59.4675             0.161764   
...                                                ...                  ...   
257              2849                         877.9760             1.000000   
                 2850                          50.1159             0.076820   
                 2855                         205.4320             0.040999   
                 2856                         809.4530             0.387449   
                 2857                         771.6110             0.370075   

                           FoxO3A_ratio_scaled  
Image_Metadata_T track_id                       
0                1                    0.614450  
                 2                    0.506309  
                 3                    0.700809  
                 4                    0.657277  
                 5                    0.718735  
...                                        ...  
257              2849                 1.000000  
                 2850                 0.356672  
                 2855                 0.065812  
                 2856                 0.608673  
                 2857                 0.948178  

[356632 rows x 7 columns]

# Processing Data

In [4]:
from scripts.Mapping_nodes_in_time import calculate_tracks_on_graphs

In [5]:
time_to_trackA_to_trackB_to_stats: dict[int, dict[int, dict[int, dict[str, float]]]] = {}

for image_T in df_pre_graphed.index.get_level_values(0).unique().to_list()[:-1]:
    time_to_trackA_to_trackB_to_stats[image_T] = calculate_tracks_on_graphs(image_T, "../data_in_processing/graphs")
    print(image_T)


0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256


In [6]:
flat = {
    (i, j, k): cols
    for i, d1 in time_to_trackA_to_trackB_to_stats.items()
    for j, d2 in d1.items()
    for k, cols in d2.items()
}

df = pd.DataFrame.from_dict(flat, orient="index")
df.index = pd.MultiIndex.from_tuples(df.index, names=["Image_Metadata_T", "track_id_t0", "track_id_t1"])

In [7]:
df

ERK_me_diff  ERK_neigh_diff
Image_Metadata_T track_id_t0 track_id_t1                             
0                1201        1495           -0.135563       -0.005412
                             874            -0.135563        0.037722
                             304            -0.135563        0.016243
                 1495        1201           -0.005412       -0.135563
                             304            -0.005412        0.016243
...                                               ...             ...
256              749         778            -0.002142        0.143538
                             1042           -0.002142        0.168321
                             2207           -0.002142       -0.056219
                             134            -0.002142       -0.032461
                             1615           -0.002142       -0.071355

[2057644 rows x 2 columns]

In [8]:
df.to_csv(f"../data_in_processing/E1S1_graphs_flattened.csv")